# Классификация ботов — решение

## Подход

**Препроцессинг.** Нормализация регистра `platform` и `event_name`, замена `iphone → ios`,
удаление полных дублей с сохранением `duplicate_ratio`, сортировка по `(cookie_id, event_ts)`.

**Признаки (54 на куку).** Четыре группы:

1. **User-Agent / устройство**: браузер, ОС, версии, `is_headless`, `is_script`,
   `ver_gap` (насколько старая версия), `os_platform_mismatch`.
2. **Временные**: `n_events`, `span_sec`, статистики интервалов (`mean/median/std/min/max/cv`),
   `share_dt0`, `share_fast`, `n_hours`, `night_share`, `dt_top_share`, `events_per_hour`.
3. **Разнообразие и состав**: уникальные объявления/категории/локации/запросы,
   `page_max/mean`, `item_ratio`, `query_ratio`, доли типов событий, `duplicate_ratio`.
4. **Курсор** (только web): `ptr_share`, `ptr_n`, `ptr_nuniq`, `ptr_uniq_ratio`,
   `ptr_x_std`, `ptr_y_std`, `ptr_step_mean/std`, `ptr_step0_share`.

**Модель.** `HistGradientBoostingClassifier` с нативными категориальными признаками
(`categorical_features=[device, ua_type, browser, os]`).



**Результаты P@R0.7 на временных срезах:**

| cutoff | n_valid | pos | P@R0.7 |
|---|---|---|---|
| 2026-04-13 | 5161 | 408 | 0.696 |
| 2026-04-17 | 1951 | 160 | 0.752 |


**Воспроизводимость.** `random_state=0`.
Python 3.12, зависимости в `requirements.txt`.

# 1. Загрузка данных и импорты

In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.ensemble import HistGradientBoostingClassifier
from metric import precision_at_recall

In [2]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


# 2. Препроцессинг ивентов

Приводим текстовые поля к нижнему регистру, убираем пробелы, объединяем признаки (`iphone → ios`, `WEB → web` и тд), сортируем по `(cookie_id, event_ts)`. 
Также думал об объединении desktop и web. Попробовал и с объединением и без, результат оставался таким же, поэтому решил не объединять.
Думал про удаление полных дубликатов на этом этапе, в итоге решил посчитать их долю для каждой куки как признак.

In [3]:
def clean_events(events):
    ev = events.copy()
    for col in ['platform', 'event_name', 'item_category',
                'item_location', 'seller_type', 'user_agent']:
        ev[col] = ev[col].str.strip()
    for col in ['platform', 'event_name', 'item_category',
                'item_location', 'seller_type']:
        ev[col] = ev[col].str.lower()
    ev['platform'] = ev.platform.replace({'iphone': 'ios', 
                                          #'desktop': 'web'
                                         })
    ev = ev.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)
    return ev

events = clean_events(pd.read_csv('data/events.csv.gz', parse_dates=['event_ts']))

## 3. Парсинг User-Agent

Разбираем строку UA на браузер, ОС, версии. Выделяем `is_headless` (HeadlessChrome) и
`is_script` (curl, scrapy, python-requests и т.п.). `ver_gap` — насколько версия браузера
отстаёт от самой свежей в тех же данных.

In [4]:
BROWSER_RULES = [                     # порядок важен: специфичные раньше общих
    ('headlesschrome', r'HeadlessChrome/(\d+)'),
    ('yabrowser',      r'YaBrowser/(\d+)'),
    ('edge',           r'Edg(?:e|A|iOS)?/(\d+)'),
    ('opera',          r'OPR/(\d+)'),
    ('samsung',        r'SamsungBrowser/(\d+)'),
    ('firefox',        r'(?:Firefox|FxiOS)/(\d+)'),
    ('chrome',         r'(?:Chrome|CriOS)/(\d+)'),   # после всех Chromium-based
    ('safari',         r'Version/(\d+)'),            # Chrome тоже пишет Safari, поэтому в конце
]

SCRIPT_RE = re.compile(r'^(go-http-client|scrapy|python-|node-fetch|curl|wget|axios|httpx|aiohttp)', re.I)

def parse_ua(ua):
    out = dict(ua_type='unknown', browser='unknown', browser_ver=np.nan,
               os='unknown', os_ver=np.nan, device_model='none')
    if not isinstance(ua, str):
        return out
    low = ua.lower()

    # 1) тип клиента и версия
    m = re.match(r'Avito/(\d+)', ua)
    if m:
        out.update(ua_type='app', browser='avito_app', browser_ver=float(m.group(1)))
    elif SCRIPT_RE.match(ua):
        out.update(ua_type='script', browser=re.split(r'[/\s]', ua)[0].lower())
    else:
        for name, pat in BROWSER_RULES:
            m = re.search(pat, ua)
            if m:
                out.update(ua_type='headless' if name == 'headlesschrome' else 'browser',
                           browser=name, browser_ver=float(m.group(1)))
                break

    # 2) ос и версия ос
    if 'android' in low:
        out['os'] = 'android'
        m = re.search(r'Android (\d+)', ua)
        out['os_ver'] = float(m.group(1)) if m else np.nan
        m = re.search(r'Android [\d.]+; ([^);]+)', ua)
        if m:
            out['device_model'] = m.group(1).split(' Build')[0].strip().lower()
    elif 'iphone' in low or 'ipad' in low:
        out['os'] = 'ios'
        m = re.search(r'i?OS (\d+)', ua)
        out['os_ver'] = float(m.group(1)) if m else np.nan
    elif 'windows' in low:
        out['os'] = 'windows'
    elif 'mac os x' in low or 'macintosh' in low:
        out['os'] = 'mac'
    elif 'linux' in low or 'x11' in low:
        out['os'] = 'linux'
    return out

uniq = pd.DataFrame({'user_agent': events.user_agent.dropna().unique()})
parsed = pd.DataFrame([parse_ua(u) for u in uniq.user_agent])
ua_table = pd.concat([uniq, parsed], axis=1)
ua_table['is_headless'] = (ua_table.ua_type == 'headless').astype(int)
ua_table['is_script']   = (ua_table.ua_type == 'script').astype(int)

# версия сравнивается только внутри одного браузера
newest = ua_table.groupby('browser').browser_ver.transform('max')
ua_table['ver_gap'] = newest - ua_table.browser_ver

ua_table = ua_table.drop(columns='device_model')

## 4. Фильтр по окну наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.

`duplicate_ratio` (доля дубликатов) тоже считается внутри окна - до удаления дублей.

In [5]:
def events_in_window(ev, meta):
    m = ev.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    m = m[(m.event_ts >= m.window_start_ts) & (m.event_ts < m.window_end_ts)]
    return m.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

ev_tr_raw = events_in_window(events, train)
ev_te_raw = events_in_window(events, test)
print(len(ev_tr_raw), len(ev_te_raw))
print('кук без событий в окне:',
      train.cookie_id.nunique() - ev_tr_raw.cookie_id.nunique(),
      test.cookie_id.nunique() - ev_te_raw.cookie_id.nunique())

198436 89690
кук без событий в окне: 0 0


## 5. Построение признаков

Функции сгруппированы по смыслу:

- `ua_features` — устройство, браузер, ОС, флаги автоматизации.
- `time_features` — интервалы, длительность, ночная активность, регулярность.
- `mix_features` — состав событий и разнообразие контента.
- `pointer_features` — поведение курсора (только web).

In [6]:
# Список типов событий берём по всем данным, чтобы колонки share_* совпадали в train и test
EVENT_NAMES = sorted(events.event_name.dropna().unique())


def mode_or_nan(s):
    m = s.dropna().mode()
    return m.iat[0] if len(m) else np.nan


# 1. User-Agent / устройство
def ua_features(ev, meta):
    ev = ev.merge(ua_table, on='user_agent', how='left')
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'device':      g.platform.agg(mode_or_nan),    # web / android / ios
        'ua_type':     g.ua_type.agg(mode_or_nan),
        'browser':     g.browser.agg(mode_or_nan),
        'os':          g.os.agg(mode_or_nan),
        'os_ver':      g.os_ver.max(),
        'ver_gap':     g.ver_gap.max(),
        'is_headless': g.is_headless.max(),
        'is_script':   g.is_script.max(),
    }).reset_index()

    WEB = ['web', 'desktop']
    # UA и платформа противоречат друг другу
    f['os_platform_mismatch'] = (
        ((f.device == 'android') & (f.os != 'android')) |
        ((f.device == 'ios')     & (f.os != 'ios')) |
        ((f.device.isin(WEB))     & f.os.isin(['android', 'ios']))
    ).astype(int)
    return f


# 2. Время между событиями 
def time_features(ev):
    ev = ev.copy()
    ev['dt'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()   # ev уже отсортирован
    ev['hour'] = ev.event_ts.dt.hour
    ev['sec_from_start'] = (ev.event_ts - ev.window_start_ts).dt.total_seconds()
    has_dt = ev.dt.notna()
    ev['is_dt0']   = (ev.dt == 0).astype(float).where(has_dt)   # то же самое время, что у прошлого события
    ev['is_fast']  = (ev.dt <= 2).astype(float).where(has_dt)   # меньше 2 секунд
    ev['is_night'] = (ev.hour < 6).astype(float)

    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events':     g.size(),
        'span_sec':     (g.event_ts.max() - g.event_ts.min()).dt.total_seconds(),
        'first_sec':    g.sec_from_start.min(),    # когда началась активность в окне
        'last_sec':     g.sec_from_start.max(),
        'dt_mean':      g.dt.mean(),
        'dt_median':    g.dt.median(),
        'dt_std':       g.dt.std(),
        'dt_min':       g.dt.min(),
        'dt_max':       g.dt.max(),
        'share_dt0':    g.is_dt0.mean(),
        'share_fast':   g.is_fast.mean(),
        'n_hours':      g.hour.nunique(),          # в скольких разных часах была активность
        'night_share':  g.is_night.mean(),
    })
    f['dt_cv'] = f.dt_std / f.dt_mean.replace(0, np.nan)    # регулярность: малое значение = ровный ритм
    f['events_per_hour'] = f.n_events / (f.span_sec / 3600).clip(lower=1/60)

    # доля самого частого интервала (у скрипта с таймером она высокая)
    cnt = ev.dropna(subset=['dt']).groupby(['cookie_id', 'dt']).size()
    top = cnt.groupby('cookie_id').agg(['max', 'sum'])
    f['dt_top_share'] = top['max'] / top['sum']
    return f.reset_index()


# 3. Состав и разнообразие событий
def mix_features(ev, dup_ratio):
    g = ev.groupby('cookie_id')
    n = g.size()

    mix = (ev.groupby(['cookie_id', 'event_name']).size().unstack(fill_value=0)
             .reindex(columns=EVENT_NAMES, fill_value=0))
    mix = mix.div(mix.sum(axis=1), axis=0).add_prefix('share_')

    div = pd.DataFrame({
        'item_nunique':  g.item_id.nunique(),
        'cat_nunique':   g.item_category.nunique(),
        'loc_nunique':   g.item_location.nunique(),
        'query_nunique': g.search_query.nunique(),
        'page_max':      g.search_page.max(),
        'page_mean':     g.search_page.mean(),
    })
    div['item_ratio']  = div.item_nunique / n      # доля уникальных объявлений
    div['query_ratio'] = div.query_nunique / n
    
    out = div.join(mix).reset_index()
    out = out.merge(dup_ratio, on='cookie_id', how='left')
    return out


# 4. Курсор (только web)
def pointer_features(ev):
    w = ev[ev.platform == 'web']
    w = w.assign(has_ptr=w.pointer_x.notna())
    f = pd.DataFrame({'ptr_share': w.groupby('cookie_id').has_ptr.mean()})

    p = w.dropna(subset=['pointer_x', 'pointer_y']).copy()
    gp = p.groupby('cookie_id')
    f['ptr_n'] = gp.size()
    f['ptr_nuniq'] = (p.drop_duplicates(['cookie_id', 'pointer_x', 'pointer_y'])
                       .groupby('cookie_id').size())
    f['ptr_uniq_ratio'] = f.ptr_nuniq / f.ptr_n          # 1 = все точки разные, мало = повторы
    f['ptr_x_std'] = gp.pointer_x.std()
    f['ptr_y_std'] = gp.pointer_y.std()

    # длина шага между соседними точками курсора
    p['step'] = np.hypot(gp.pointer_x.diff(), gp.pointer_y.diff())
    p['step0'] = (p.step == 0).astype(float).where(p.step.notna())
    gp = p.groupby('cookie_id')
    f['ptr_step_mean'] = gp.step.mean()
    f['ptr_step_std']  = gp.step.std()
    f['ptr_step0_share'] = gp.step0.mean()               # доля «не сдвинулся»
    return f.reset_index()

## 6. Сборка таблицы признаков

Собираем все группы через `merge` по `cookie_id`. Счётчики курсора у web-кук без координат
заполняем нулём — это настоящий 0, а не пропуск. У мобильных остаётся NaN.

В конце выравниваем порядок строк под `train`/`test`, чтобы `X` и `y` совпадали.

In [7]:
def build_features(ev, meta, dup_ratio):
    F = ua_features(ev, meta)                 # уже содержит все куки из meta
    for part in [time_features(ev), mix_features(ev, dup_ratio), pointer_features(ev)]:
        F = F.merge(part, on='cookie_id', how='left')

    F['n_events'] = F['n_events'].fillna(0)   # на случай кук без событий в окне (в test)

    # счётчики курсора: у веб-кук без координат это настоящий 0, у мобильных остаётся NaN
    web = F.device.eq('web')
    for c in ['ptr_n', 'ptr_nuniq']:
        F.loc[web, c] = F.loc[web, c].fillna(0)
    F = meta[['cookie_id']].merge(F, on='cookie_id', how='left')
    return F

# считаем дубли внутри окна
def dup_ratio_fn(ev):
    return (ev.groupby('cookie_id')
              .apply(lambda d: d.duplicated().mean(), include_groups=False)
              .rename('duplicate_ratio')
              .reset_index())

dup_tr = dup_ratio_fn(ev_tr_raw)
dup_te = dup_ratio_fn(ev_te_raw)

# и только теперь дедуп
ev_tr = ev_tr_raw.drop_duplicates().reset_index(drop=True)
ev_te = ev_te_raw.drop_duplicates().reset_index(drop=True)

Ftr = build_features(ev_tr, train, dup_tr)
Fte = build_features(ev_te, test,  dup_te)

assert len(Ftr) == len(train)
assert (Ftr.cookie_id.values == train.cookie_id.values).all()
assert (Fte.cookie_id.values == test.cookie_id.values).all()

ytr = train.target.values
print(Ftr.shape, Fte.shape)
print('кук без событий в окне, train/test:',
      (Ftr.n_events == 0).sum(), (Fte.n_events == 0).sum())

# доли пропусков в test должны быть похожи на train
cmp = pd.concat([Ftr.isna().mean(), Fte.isna().mean()], axis=1, keys=['train', 'test']).round(2)
print(cmp[(cmp > 0).any(axis=1)])
T = Ftr.merge(train[['cookie_id', 'target']], on='cookie_id')
cols = ['n_events', 'dt_median', 'dt_cv', 'share_fast', 'night_share', 'item_ratio', 'ptr_share']
print(T.groupby('target')[cols].median().T)

(11091, 54) (4909, 54)
кук без событий в окне, train/test: 0 0
                 train  test
os_ver            0.53  0.54
ver_gap           0.02  0.01
dt_mean           0.02  0.02
dt_median         0.02  0.02
dt_std            0.06  0.06
dt_min            0.02  0.02
dt_max            0.02  0.02
share_dt0         0.02  0.02
share_fast        0.02  0.02
dt_cv             0.06  0.06
dt_top_share      0.02  0.02
page_max          0.08  0.09
page_mean         0.08  0.09
ptr_share         0.47  0.46
ptr_n             0.48  0.47
ptr_nuniq         0.48  0.47
ptr_uniq_ratio    0.64  0.63
ptr_x_std         0.66  0.65
ptr_y_std         0.66  0.65
ptr_step_mean     0.66  0.65
ptr_step_std      0.68  0.68
ptr_step0_share   0.66  0.65
target               0          1
n_events     12.000000  20.000000
dt_median    60.500000  25.000000
dt_cv         2.044825   2.596203
share_fast    0.000000   0.000000
night_share   0.000000   0.000000
item_ratio    0.527778   0.500000
ptr_share     0.941176   0.00000

## 7. Кодирование категориальных признаков

Категории кодируем согласованно в train и test — `pd.Categorical` по объединённым данным.
HGB получает их как числовые коды и сам разбирается с нелинейностями.

In [8]:
cat_cols = ['device', 'ua_type', 'browser', 'os']

# склеиваем train+test, чтобы коды категорий совпадали
X_all = pd.concat(
    [Ftr.drop(columns='cookie_id'), Fte.drop(columns='cookie_id')],
    axis=0, ignore_index=True
)

for c in cat_cols:
    # объединяем train и test, чтобы коды совпадали
    cats = pd.Categorical(pd.concat([Ftr[c], Fte[c]]).astype(str)).categories
    X_all[c] = pd.Categorical(X_all[c].astype(str), categories=cats).codes

n_tr = len(Ftr)
Xtr_enc = X_all.iloc[:n_tr].reset_index(drop=True)
Xte_enc = X_all.iloc[n_tr:].reset_index(drop=True)

# индексы категориальных колонок для HGB
cat_idx = [Xtr_enc.columns.get_loc(c) for c in cat_cols]

ytr = train.target.values
print(Xtr_enc.shape, Xte_enc.shape, 'cat_idx =', cat_idx)

(11091, 53) (4909, 53) cat_idx = [0, 1, 2, 3]


## 8. Валидация

### Схема валидации

Тест лежит позже train, поэтому используем временную валидацию: `window_start_ts >= cutoff`.

Одиночный срез шумит, поэтому смотрим сразу несколько. Ранние срезы (04-08, 04-10)
дают мало данных для обучения — метрика там занижена и служит только для контроля
динамики. Основные ориентиры — 04-13 и 04-17.

### Результаты по срезам (HGB, `max_iter=500, lr=0.05, leaves=31, min_samples_leaf=20, l2=1.0`)

| cutoff | n_valid | pos (ботов) | P@R0.7 |
|---|---|---|---|
| 2026-04-08 | 9522 | 772 | 0.538 |
| 2026-04-10 | 7786 | 636 | 0.606 |
| 2026-04-13 | 5161 | 408 | 0.696 |
| 2026-04-17 | 1951 | 160 | 0.752 |

Почему растёт к поздним срезам: чем позже cutoff, тем больше данных остаётся в обучении
(на 04-08 — 1569 кук, на 04-17 — 9140).

### Что пробовал и почему остановился на HGB

| Модель / конфигурация | 04-13 | 04-17 | Среднее |
|---|---|---|---|
| **HGB (baseline)** | **0.696** | **0.752** | **0.724** |
| HGB, `lr=0.03, max_iter=1500` | 0.665 | 0.778 | 0.721 |
| HGB, `max_leaf_nodes=15, l2=3.0` | 0.684 | 0.742 | 0.713 |
| HGB, `max_leaf_nodes=63, l2=3.0` | 0.688 | 0.727 | 0.707 |
| LightGBM, `n_estimators=2000` | 0.664 | 0.723 | 0.693 |
| CatBoost, `auto_class_weights='Balanced'` | 0.644 | 0.675 | 0.660 |

**Вывод:**

- HGB с дефолтными гиперпараметрами даёт лучшее среднее по двум срезам и меньший
  разброс. Конфигурации, которые выигрывали на одном срезе, проигрывали на другом — это
  признак подгонки под конкретный cutoff, а не реального улучшения.
- LightGBM и CatBoost оказались хуже на обоих срезах. Использовать их в одиночку смысла нет.
- Итоговая модель — HGB с `max_iter=500, learning_rate=0.05, max_leaf_nodes=31,
  min_samples_leaf=20, l2_regularization=1.0, categorical_features=[device, ua_type,
  browser, os], random_state=0`.


### Ограничения валидации

- На 04-17 всего 160 ботов — разброс метрики ±0.05.
- Тест лежит ещё позже 04-17, поведение ботов могло сдвинуться. Реальный скор на тесте
  может быть ниже локального.

In [9]:
cutoff = '2026-04-17'
is_valid = train.window_start_ts >= cutoff

model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    categorical_features=cat_idx,
    random_state=0,
)
model.fit(Xtr_enc.loc[~is_valid], ytr[~is_valid])
p_va = model.predict_proba(Xtr_enc.loc[is_valid])[:, 1]

print('валид. кук:', is_valid.sum(), '| ботов в валид.:', ytr[is_valid].sum())
print('P@R0.7:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('уникальных score:', pd.Series(p_va).nunique(), 'из', len(p_va))

валид. кук: 1951 | ботов в валид.: 160
P@R0.7: 0.7517
уникальных score: 1951 из 1951


In [10]:
for cutoff in ['2026-04-08', '2026-04-10', '2026-04-13', '2026-04-17']:
    is_valid = train.window_start_ts >= cutoff
    if ytr[is_valid].sum() < 20:
        print(f'{cutoff}: пропуск (мало ботов)')
        continue

    m = HistGradientBoostingClassifier(
        max_iter=500, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=20, l2_regularization=1.0,
        categorical_features=cat_idx, random_state=0,
    )
    m.fit(Xtr_enc[~is_valid.values], ytr[~is_valid.values])
    p = m.predict_proba(Xtr_enc[is_valid.values])[:, 1]
    print(f'{cutoff}: P@R0.7 = {precision_at_recall(ytr[is_valid.values], p):.4f}  '
          f'(n={is_valid.sum()}, pos={ytr[is_valid].sum()})')

2026-04-08: P@R0.7 = 0.5599  (n=9522, pos=772)
2026-04-10: P@R0.7 = 0.6073  (n=7786, pos=636)
2026-04-13: P@R0.7 = 0.6698  (n=5161, pos=408)
2026-04-17: P@R0.7 = 0.7517  (n=1951, pos=160)


## 9. Финальная модель и сабмит

Обучаемся на train, предсказываем test. Формат: `cookie_id, score` (score ∈ [0, 1]).

Проверки:
- число строк совпадает с `test.csv`;
- `cookie_id` уникальны;
- `score` в диапазоне.

In [11]:
model_final = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    categorical_features=cat_idx,
    random_state=0,
)
model_final.fit(Xtr_enc, ytr)
p_te = model_final.predict_proba(Xte_enc)[:, 1]

sub = pd.DataFrame({
    'cookie_id': Fte.cookie_id.values,
    'score': p_te,
})
assert len(sub) == len(test), 'несовпадение строк'
assert sub.cookie_id.is_unique, 'дубликаты'
assert sub.score.between(0, 1).all(), 'score вне [0,1]'

sub.to_csv('submission.csv', index=False)
print(sub.shape, '| уникальных score:', sub.score.nunique())
print(sub.head())

(4909, 2) | уникальных score: 4908
             cookie_id     score
0  ck_315fb710a0e371e7  0.003067
1  ck_a76ee3b3e3e522fd  0.046998
2  ck_94c9a4d382689e82  0.011324
3  ck_8eaf9509ad9462a0  0.003260
4  ck_9a88a5a989cb5bc6  0.012908


In [12]:
print(sub.score.describe())
print('score > 0.5:', (sub.score > 0.5).sum())
print('score > 0.3:', (sub.score > 0.3).sum())
print('score > 0.1:', (sub.score > 0.1).sum())

count    4909.000000
mean        0.082567
std         0.205317
min         0.001419
25%         0.006280
50%         0.013738
75%         0.035005
max         0.993062
Name: score, dtype: float64
score > 0.5: 302
score > 0.3: 392
score > 0.1: 642
